In [46]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [ ]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://opencode.ai/zen/v1/"
)

In [48]:
def llm(prompt):
    response = openai_client.chat.completions.create(
        model="deepseek-v4-flash-free",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

In [49]:
llm("Hey, what's up?")

"Hey! Not much on this end, just hanging out in the digital realm ready to help out. What's going on with you?"

In [50]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

That's exciting! I'd be happy to help you figure that out.

To give you the most accurate answer, could you let me know the **name of the course** you discovered? Many online courses (like those on Coursera, edX, or YouTube) are self-paced, which means you can join anytime. Other courses, especially live cohorts or university programs, have specific enrollment windows.

Once I know which course it is, I can help you look up whether enrollment is currently open or guide you to the right page to check.


In [51]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [52]:
answer = llm(prompt)
print(answer)

Based on the context, yes, you can join now. The course says: "I just discovered the course. Can I still join? Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions." It also states that you can start learning and submitting homework without registering.


## Dataset

In [53]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

## Search

In [54]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

## Building prompt

In [55]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [56]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [57]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [58]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [59]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

# The llm 

In [60]:
# we use chat.completions (as we have opencode go)

response = openai_client.chat.completions.create(
    model="deepseek-v4-flash-free",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

In [61]:
response


ChatCompletion(id='56cb3dde-d805-4d58-8e94-5b4a0aad4819', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Yes, based on the context provided, you can join now. The FAQ directly answers this question with: \n\n> **Yes**, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nTo give you the full picture based on the other FAQs:\n\n* **Registration:** You don\'t need to register first. You can just start learning and submitting homework while the form is open.\n* **Certificate:** You can only earn a certificate by participating in a "live" cohort and submitting your Capstone project while the peer-review window is open. The self-paced mode does not offer a certificate.\n* **Next Cohort:** If the project submission window for the current cohort has already closed, the course will be offered live again in **Summer 2025**.', refusal=None, role='assistant', annotations=None, au

In [62]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.total_tokens * output_price
)

cost

0.008019

In [ ]:
def llm(instructions, user_prompt, model="deepseek-v4-flash-free"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [ ]:
def rag(query, model="deepseek-v4-flash-free"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [68]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)


Yes, you can join now. According to the course FAQ, you can still join, but if you want to receive a certificate, you need to make sure you submit your project while submissions are still being accepted.


# rag helper

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI
import os

documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://opencode.ai/zen/v1/"
)

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join. If you want to receive a certificate, you must submit your project while submissions are still being accepted.
